# Transcribe audio with OpenAI's Whisper API in Google Colab

Companion notebook for the MLforSEO tutorial
[How to transcribe audio with OpenAI's Whisper API in Google Colab (Python)](https://www.mlforseo.com/machine-learning-tutorials/audio-transcription/how-to-transcribe-audio-with-openais-whisper-api-in-google-colab-python/).

- **Single file:** steps 1-6 upload one audio file and print its transcript.
- **Files over 25 MB:** split them into chunks with `pydub` and transcribe each chunk, or export SRT captions.
- **Bulk:** upload several files and download one `.txt` transcript per file.

The API accepts `mp3`, `mp4`, `mpeg`, `mpga`, `m4a`, `wav` and `webm` files up to 25 MB.

**2026 refresh:** the original notebook mixed the old `openai.api_key` pattern with the new client and failed with a `NameError`. This version uses the current SDK (`from openai import OpenAI`) and reads your key from a Colab secret.

## 2. Install the required libraries
`openai` is the official SDK. `pydub` is only needed for files over 25 MB (Colab already has `ffmpeg`).

In [ ]:
%pip install -q --upgrade openai pydub

## 3. Authenticate with your API key
Add your key as a Colab secret: click the key icon in the left sidebar, add a secret named `OPENAI_API_KEY`, paste the key as its value and switch on notebook access.
Running locally? Set the `OPENAI_API_KEY` environment variable instead. Never paste the key into the notebook.

In [ ]:
import os
from openai import OpenAI

# Read the key from a Colab secret (or from the OPENAI_API_KEY environment variable outside Colab)
try:
    from google.colab import userdata
    os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')
except ImportError:
    pass  # not in Colab: export OPENAI_API_KEY in your shell before starting Python

client = OpenAI()  # picks up OPENAI_API_KEY automatically

## 4. Upload an audio file

In [ ]:
from google.colab import files

# Upload audio file
uploaded = files.upload()

file_path = next(iter(uploaded))

## 5. Define the transcription function
`whisper-1` is the Whisper model the tutorial uses. `gpt-4o-transcribe` and `gpt-4o-mini-transcribe` are newer speech-to-text models on the same endpoint; they return text only (no SRT or timestamps).

In [ ]:
MODEL = 'whisper-1'  # or 'gpt-4o-transcribe' / 'gpt-4o-mini-transcribe'
MAX_FILE_SIZE = 25 * 1024 * 1024  # the API rejects uploads over 25 MB

def transcribe_audio(file_path):
    if os.path.getsize(file_path) > MAX_FILE_SIZE:
        raise ValueError(f'{file_path} is over 25 MB: compress it, or split it with transcribe_long_audio() from the GitHub notebook.')
    with open(file_path, 'rb') as audio_file:
        response = client.audio.transcriptions.create(
            model=MODEL,
            file=audio_file
        )
    return response.text

## 6. Call the function

In [ ]:
# Transcribe the uploaded audio file
transcription = transcribe_audio(file_path)
print(transcription)

## Optional: files over 25 MB, and SRT captions
`split_audio` cuts a long recording into 10-minute mono MP3 chunks (about 5 MB each). `transcribe_long_audio` transcribes the chunks in order and joins the text.

`transcribe_to_srt` uses `whisper-1`'s segment timestamps to build an `.srt` caption file, shifting each chunk's timestamps so they line up with the full recording.

In [ ]:
from pydub import AudioSegment

CHUNK_MINUTES = 10

def split_audio(file_path, chunk_minutes=CHUNK_MINUTES):
    # Returns [(chunk_path, start_seconds), ...]
    audio = AudioSegment.from_file(file_path).set_channels(1)
    chunk_ms = int(chunk_minutes * 60 * 1000)
    base = os.path.splitext(os.path.basename(file_path))[0]
    chunks = []
    for i, start in enumerate(range(0, len(audio), chunk_ms), start=1):
        chunk_path = f'{base}_part{i:03d}.mp3'
        audio[start:start + chunk_ms].export(chunk_path, format='mp3', bitrate='64k')
        chunks.append((chunk_path, start / 1000))
    return chunks

def transcribe_long_audio(file_path):
    if os.path.getsize(file_path) <= MAX_FILE_SIZE:
        return transcribe_audio(file_path)
    return ' '.join(transcribe_audio(chunk_path) for chunk_path, _ in split_audio(file_path))

def srt_timestamp(seconds):
    ms = int(round(seconds * 1000))
    h, ms = divmod(ms, 3_600_000)
    m, ms = divmod(ms, 60_000)
    s, ms = divmod(ms, 1000)
    return f'{h:02d}:{m:02d}:{s:02d},{ms:03d}'

def transcribe_to_srt(file_path):
    parts = split_audio(file_path) if os.path.getsize(file_path) > MAX_FILE_SIZE else [(file_path, 0.0)]
    entries = []
    for part_path, offset in parts:
        with open(part_path, 'rb') as audio_file:
            response = client.audio.transcriptions.create(
                model='whisper-1',  # segment timestamps are a whisper-1 feature
                file=audio_file,
                response_format='verbose_json',
                timestamp_granularities=['segment'],
            )
        for seg in response.segments:
            entries.append((seg.start + offset, seg.end + offset, seg.text.strip()))
    return '\n'.join(
        f'{i}\n{srt_timestamp(start)} --> {srt_timestamp(end)}\n{text}\n'
        for i, (start, end, text) in enumerate(entries, start=1)
    )

# Example: captions for the file uploaded in step 4
# srt_path = os.path.splitext(file_path)[0] + '.srt'
# with open(srt_path, 'w', encoding='utf-8') as f:
#     f.write(transcribe_to_srt(file_path))
# files.download(srt_path)

# Transcription of multiple files (in bulk)
Run steps 2, 3 and 5 first. The cell below uploads several files, transcribes each one and saves a `.txt` file with the same base name, then downloads it.
Files over 25 MB are skipped with a message; run them through `transcribe_long_audio()` above.

In [ ]:
from google.colab import files

# Upload multiple audio files
uploaded = files.upload()

# Iterate over the uploaded files and transcribe each one
for file_name in uploaded:
    if os.path.getsize(file_name) > MAX_FILE_SIZE:
        print(f'Skipped {file_name}: over 25 MB (use transcribe_long_audio)')
        continue
    transcription = transcribe_audio(file_name)

    # Write the transcription to a text file
    output_file_name = f'{os.path.splitext(file_name)[0]}.txt'
    with open(output_file_name, 'w', encoding='utf-8') as output_file:
        output_file.write(transcription)

    print(f'Transcription saved to {output_file_name}')

    # Download the text file
    files.download(output_file_name)